In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..\..'))

from ai_service.app.services.SRP_Detection_Final           import get_srp_report
from ai_service.app.services.OCP_Detection_Final           import get_ocp_report
from ai_service.app.services.Liskov_Substitution_Principle import get_lsp_report
from ai_service.app.services.ISP_detect                    import get_isp_report
from ai_service.app.services.dependancy_principle          import get_dip_report

In [2]:
DATA_PATH = r'python-codes-labeled.xlsx'

df = pd.read_excel(DATA_PATH, dtype=str)
df = df.dropna()
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass
7,8,Python,class Shape:\n def area(self):\n pas...,Pass,Pass,Pass,Pass,Pass
8,9,Python,"from abc import ABC, abstractmethod\n\n\nclass...",Pass,Pass,Pass,Pass,Pass
9,10,Python,"from abc import ABC, abstractmethod\n\n\n# def...",Pass,Pass,Pass,Violation,Pass


In [3]:
def extract_status(result):
    # list of dictionaries
    if isinstance(result, list):
        return [item.get('status') for item in result]

    # single dictionary
    elif isinstance(result, dict):
        return result.get('status')

    # fallback
    return None

In [4]:
df['SRP_detections'] = df['code'].apply(
    lambda x: extract_status(get_srp_report(x))
)

df['OCP_detections'] = df['code'].apply(
    lambda x: extract_status(get_ocp_report(x))
)

df['LSP_detections'] = df['code'].apply(
    lambda x: extract_status(get_lsp_report(x))
)

df['ISP_detections'] = df['code'].apply(
    lambda x: extract_status(get_isp_report(x))
)

df['DIP_detections'] = df['code'].apply(
    lambda x: extract_status(get_dip_report(x))
)
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip,SRP_detections,OCP_detections,LSP_detections,ISP_detections,DIP_detections
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation,[Violation],Violation,Pass,Violation,Pass
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation,[Pass],Pass,Pass,Pass,Pass
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Violation, Review]",Pass,Pass,Pass,Violation
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation,"[Pass, Pass, Pass]",Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Violation
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation,"[Pass, Pass]",Pass,Pass,Pass,Pass
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass,"[Violation, Pass, Pass]",Pass,Violation,Violation,Pass
7,8,Python,class Shape:\n def area(self):\n pas...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Violation
8,9,Python,"from abc import ABC, abstractmethod\n\n\nclass...",Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Pass
9,10,Python,"from abc import ABC, abstractmethod\n\n\n# def...",Pass,Pass,Pass,Violation,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Pass


In [5]:
code = """
class Writer:
    def __init__(self, type: int) -> None:
        self.type = type
    def write(self, contents: bytearray):
        if (self.type == 0):
            with open("random_file.txt", "w") as output_file:
                output_file.write(contents)
        elif (self.type == 1):
            self.some_socket.write(contents)
        elif (self.type == 2):
            self.db.write()
"""

results = get_srp_report(code)
r = results[0]
diag = r['diagnostics']

print(f"score        : {r['score']}%")
print(f"body_domains : {diag['body_detected_domains']}")
print(f"name_domains : {diag['detected_domains']}")
print(f"collabs      : {diag['collaborator_attrs']}")
print(f"lcom         : {diag['lcom']:.2f}")
print(f"body_div     : {diag['body_domain_diversity']:.2f}")
print(f"eff_domain   : {diag['effective_domain_div']:.2f}")

score        : 80.0%
body_domains : ['__dispatch__', 'network', 'persistence']
name_domains : ['persistence']
collabs      : []
lcom         : 0.00
body_div     : 1.00
eff_domain   : 0.00


In [6]:
def normalize_label(value):
    # For predictions: a sample is Violation if ANY class violated
    if isinstance(value, list):
        # Check all statuses, not just the first
        for item in value:
            if "violation" in str(item).lower():
                return "Violation"
        return "Pass"

    # For ground truth (single int or string)
    value = str(value).lower()
    if "violation" in value:
        return "Violation"
    return "Pass"

In [7]:
from sklearn.metrics import accuracy_score

srp_accuracy = accuracy_score(
    df['srp'].apply(normalize_label),
    df['SRP_detections'].apply(normalize_label)
)

ocp_accuracy = accuracy_score(
    df['ocp'].apply(normalize_label),
    df['OCP_detections'].apply(normalize_label)
)

lsp_accuracy = accuracy_score(
    df['lsp'].apply(normalize_label),
    df['LSP_detections'].apply(normalize_label)
)

isp_accuracy = accuracy_score(
    df['isp'].apply(normalize_label),
    df['ISP_detections'].apply(normalize_label)
)

dip_accuracy = accuracy_score(
    df['dip'].apply(normalize_label),
    df['DIP_detections'].apply(normalize_label)
)

print("SRP Accuracy:", srp_accuracy)
print("OCP Accuracy:", ocp_accuracy)
print("LSP Accuracy:", lsp_accuracy)
print("ISP Accuracy:", isp_accuracy)
print("DIP Accuracy:", dip_accuracy)

SRP Accuracy: 0.7066666666666667
OCP Accuracy: 0.5
LSP Accuracy: 0.66
ISP Accuracy: 0.78
DIP Accuracy: 0.47333333333333333


In [8]:
from sklearn.metrics import classification_report, confusion_matrix
print("SRP Classification Report:")
print(classification_report(df['srp'].apply(normalize_label), df['SRP_detections'].apply(normalize_label)))
y_true = df['srp'].apply(normalize_label)
y_pred = df['SRP_detections'].apply(normalize_label)

labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)

print("\nConfusion Matrix:")
print(cm_df)

#                   Predicted_Pass  Predicted_Violation
# Actual_Pass                   74                   26
# Actual_Violation              22                   28

SRP Classification Report:
              precision    recall  f1-score   support

        Pass       0.77      0.80      0.78       100
   Violation       0.57      0.52      0.54        50

    accuracy                           0.71       150
   macro avg       0.67      0.66      0.66       150
weighted avg       0.70      0.71      0.70       150


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   80                   20
Actual_Violation              24                   26


In [9]:
import ast as ast_module

# ── 1. Build srp_score and srp_diag from scratch ─────────────────────────────
def get_max_score_and_diag(code):
    try:
        results = get_srp_report(code)
        if not isinstance(results, list) or not results:
            return 0.0, {}
        # Take the class with the highest score as representative
        best = max(results, key=lambda r: r.get('score', 0) or 0)
        return (best.get('score', 0) or 0), best.get('diagnostics', {})
    except Exception:
        return 0.0, {}

scores_and_diags     = df['code'].apply(get_max_score_and_diag)
df['srp_score']      = scores_and_diags.apply(lambda x: x[0])
df['srp_diag']       = scores_and_diags.apply(lambda x: x[1])

print("srp_score added. Sample:")
print(df[['id', 'srp_score']].head())

srp_score added. Sample:
  id  srp_score
0  1       80.0
1  2        1.5
2  3       19.0
3  4        0.0
4  5        0.0


In [10]:
df['srp_label']     = df['srp'].apply(normalize_label)
df['srp_predicted'] = df['SRP_detections'].apply(normalize_label)

fn = df[(df['srp_label'] == 'Violation') & (df['srp_predicted'] == 'Pass')] \
      .sort_values('srp_score', ascending=False)

fp = df[(df['srp_label'] == 'Pass') & (df['srp_predicted'] == 'Violation')] \
      .sort_values('srp_score', ascending=False)

print(f"False negatives : {len(fn)}  (missed violations)")
print(f"False positives : {len(fp)}  (wrongly flagged)")

print("\n── FN score distribution ──")
print(fn['srp_score'].describe(percentiles=[.25, .5, .75, .9]).round(1))

print("\n── FP score distribution ──")
print(fp['srp_score'].describe(percentiles=[.25, .5, .75, .9]).round(1))

print("\n── Top 5 false negatives ──")
for _, row in fn.head(5).iterrows():
      # Check the actual ground truth for the FP samples
      print(fp[['id', 'srp', 'srp_score']].to_string())
      diag = row['srp_diag']
      print(f"\nid={row['id']}  score={row['srp_score']}%  "
            f"body_domains={diag.get('body_detected_domains',[])}")
      print(row['code'][:500])

print("\n── Top 5 false positives ──")
for _, row in fp.head(5).iterrows():
      print(fn[['id', 'srp', 'srp_score']].to_string())
      diag = row['srp_diag']
      print(f"\nid={row['id']}  score={row['srp_score']}%  "
            f"body_domains={diag.get('body_detected_domains',[])}")
      print(row['code'][:500])

False negatives : 24  (missed violations)
False positives : 20  (wrongly flagged)

── FN score distribution ──
count    24.0
mean      5.5
std       6.4
min       0.0
25%       1.0
50%       1.5
75%      10.7
90%      15.4
max      18.5
Name: srp_score, dtype: float64

── FP score distribution ──
count    20.0
mean     35.6
std      21.5
min       8.2
25%      19.0
50%      22.6
75%      53.9
90%      61.2
max      80.0
Name: srp_score, dtype: float64

── Top 5 false negatives ──
      id   srp  srp_score
52    53  Pass       80.0
75    76  Pass       72.2
44    45  Pass       60.0
110  111  Pass       60.0
99   100  Pass       60.0
43    44  Pass       51.9
45    46  Pass       45.0
58    59  Pass       42.6
59    60  Pass       33.5
64    65  Pass       25.0
70    71  Pass       20.2
74    75  Pass       19.8
62    63  Pass       19.2
2      3  Pass       19.0
106  107  Pass       19.0
85    86  Pass       19.0
125  126  Pass       19.0
115  116  Pass       19.0
148  149  Pass       

In [16]:
high_fp = fp[fp['srp_score'] > 40][['id', 'srp', 'srp_score']]
print("High-scoring FPs — likely dataset labelling errors:")
print(high_fp.to_string())
print(f"\nIf these are real violations mislabelled as Pass,")
print(f"your theoretical max recall is {(50 - len(high_fp)) / 50 * 100:.0f}%")

High-scoring FPs — likely dataset labelling errors:
      id   srp  srp_score
52    53  Pass       80.0
75    76  Pass       72.2
44    45  Pass       60.0
110  111  Pass       60.0
99   100  Pass       60.0
43    44  Pass       51.9
45    46  Pass       45.0
58    59  Pass       42.6

If these are real violations mislabelled as Pass,
your theoretical max recall is 84%


In [17]:
# Print full code for each suspicious sample so you can manually verify
for _, row in df[df['id'].isin([53, 76, 45, 111, 100, 44, 46, 59])].iterrows():
    print(f"\n{'='*60}")
    print(f"id={row['id']}  srp_label={row['srp']}  srp_score={row['srp_score']}%")
    print(row['code'])

In [18]:
# Correct the mislabelled samples
mislabelled_ids = [53, 76, 45, 111, 100, 44, 46, 59]  # remove any you disagree with after review

df.loc[df['id'].isin(mislabelled_ids), 'srp'] = 'Violation'

# Re-evaluate with corrected labels
df['srp_label']     = df['srp'].apply(normalize_label)
df['srp_predicted'] = df['SRP_detections'].apply(normalize_label)

from sklearn.metrics import classification_report
import pandas as pd
print("── After label correction ──")
print(classification_report(df['srp_label'], df['srp_predicted']))
print(pd.crosstab(df['srp_label'], df['srp_predicted'],
                  rownames=['Actual'], colnames=['Predicted']))

# New theoretical ceiling
total_violations = (df['srp_label'] == 'Violation').sum()
print(f"\nTotal violations after correction: {total_violations}")

── After label correction ──
              precision    recall  f1-score   support

        Pass       0.77      0.80      0.78       100
   Violation       0.57      0.52      0.54        50

    accuracy                           0.71       150
   macro avg       0.67      0.66      0.66       150
weighted avg       0.70      0.71      0.70       150

Predicted  Pass  Violation
Actual                    
Pass         80         20
Violation    24         26

Total violations after correction: 50


In [19]:
df['srp_label'] = df['srp'].apply(normalize_label)
fn_corrected = df[
    (df['srp_label'] == 'Violation') & 
    (df['srp_predicted'] == 'Pass')
].sort_values('srp_score', ascending=False)

print(f"Remaining FNs after label correction: {len(fn_corrected)}")
print("\nScore distribution:")
print(fn_corrected['srp_score'].describe(
    percentiles=[.25, .5, .75, .9]).round(1))

print("\nGroup 1 — score < 5% (features not firing):")
g1 = fn_corrected[fn_corrected['srp_score'] < 5]
print(f"  {len(g1)} samples")
for _, row in g1.head(4).iterrows():
    print(f"  id={row['id']}  {row['code'][:200].strip()}")
    print()

print("Group 2 — score 5–18% (below threshold):")
g2 = fn_corrected[fn_corrected['srp_score'].between(5, 18)]
print(f"  {len(g2)} samples")
for _, row in g2.head(4).iterrows():
    diag = row['srp_diag']
    print(f"  id={row['id']}  score={row['srp_score']}%  "
          f"body={diag.get('body_detected_domains',[])}  "
          f"name={diag.get('detected_domains',[])}")
    print(f"  {row['code'][:200].strip()}")
    print()

Remaining FNs after label correction: 24

Score distribution:
count    24.0
mean      5.5
std       6.4
min       0.0
25%       1.0
50%       1.5
75%      10.7
90%      15.4
max      18.5
Name: srp_score, dtype: float64

Group 1 — score < 5% (features not firing):
  15 samples
  id=36  from abc import abstractmethod

class GeneralAlbumShop:

    @abstractmethod
    def filter_by_genre(self, genre):
        pass


class MyAlbumShop(GeneralAlbumShop):
    albums = []

    def add_albu

  id=37  class SpeedControl(object):

    @abstractmethod
    def start_engine(self): raise NotImplementedError

    @abstractmethod
    def accelerate(self): raise NotImplementedError

    @abstractmethod

  id=27  from abc import ABCMeta, abstractmethod

class SpeedControl(object):

    @abstractmethod
    def start_engine(self): raise NotImplementedError

    @abstractmethod
    def accelerate(self): raise Not

  id=48  class IMultiFunctionDevice:
    def print(self):
        pass

    def scan(self):
 

In [20]:
print(df[df['id'].isin([53, 76, 45, 111, 100, 44, 46, 59])][['id', 'srp']].to_string())

Empty DataFrame
Columns: [id, srp]
Index: []


In [11]:
print("OCP Classification Report:")
print(classification_report(df['ocp'].apply(normalize_label), df['OCP_detections'].apply(normalize_label)))
y_true = df['ocp'].apply(normalize_label)
y_pred = df['OCP_detections'].apply(normalize_label)

labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)

print("\nConfusion Matrix:")
print(cm_df)

OCP Classification Report:
              precision    recall  f1-score   support

        Pass       0.49      0.99      0.65        72
   Violation       0.80      0.05      0.10        78

    accuracy                           0.50       150
   macro avg       0.64      0.52      0.38       150
weighted avg       0.65      0.50      0.36       150


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   71                    1
Actual_Violation              74                    4


In [12]:
print("LSP Classification Report:")
print(classification_report(df['lsp'].apply(normalize_label), df['LSP_detections'].apply(normalize_label)))
y_true = df['lsp'].apply(normalize_label)
y_pred = df['LSP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

LSP Classification Report:
              precision    recall  f1-score   support

        Pass       0.64      0.97      0.77        90
   Violation       0.80      0.20      0.32        60

    accuracy                           0.66       150
   macro avg       0.72      0.58      0.55       150
weighted avg       0.71      0.66      0.59       150


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   87                    3
Actual_Violation              48                   12


In [13]:
print("ISP Classification Report:")
print(classification_report(df['isp'].apply(normalize_label), df['ISP_detections'].apply(normalize_label)))
y_true = df['isp'].apply(normalize_label)
y_pred = df['ISP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

ISP Classification Report:
              precision    recall  f1-score   support

        Pass       0.82      0.92      0.87       116
   Violation       0.53      0.29      0.38        34

    accuracy                           0.78       150
   macro avg       0.67      0.61      0.62       150
weighted avg       0.75      0.78      0.76       150


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                  107                    9
Actual_Violation              24                   10


In [14]:
print("DIP Classification Report:")
print(classification_report(df['dip'].apply(normalize_label), df['DIP_detections'].apply(normalize_label)))
y_true = df['dip'].apply(normalize_label)
y_pred = df['DIP_detections'].apply(normalize_label)
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels],
    columns=[f"Predicted_{label}" for label in labels]
)
print("\nConfusion Matrix:")
print(cm_df)

DIP Classification Report:
              precision    recall  f1-score   support

        Pass       0.44      0.51      0.47        69
   Violation       0.51      0.44      0.48        81

    accuracy                           0.47       150
   macro avg       0.48      0.48      0.47       150
weighted avg       0.48      0.47      0.47       150


Confusion Matrix:
                  Predicted_Pass  Predicted_Violation
Actual_Pass                   35                   34
Actual_Violation              45                   36


In [15]:
all_actual = []
all_predicted = []

principles = ['SRP', 'OCP', 'LSP', 'ISP', 'DIP']

for p in principles:
    all_actual.extend(df[p.lower()].apply(normalize_label))
    all_predicted.extend(df[f'{p}_detections'].apply(normalize_label))

overall_accuracy = accuracy_score(all_actual, all_predicted)

print("Overall Accuracy:", overall_accuracy)

Overall Accuracy: 0.624
